# SST-2 (BERT): complete the record to full grid parity (N=1000)

SST-2 with BERT was the original pilot run. Its quantum kernel matrix already
exists (`kernel_matrices_week3.npz`), so this notebook does **not** recompute
the 83-minute kernel. Instead it reconstructs the exact test split, verifies it
reproduces the saved kernel, then computes every field the other 30
configurations carry: all diagnostics, the lambda sweep, the proxy, off-diagonal
means, the bootstrap CI, and a matched learning curve.

**What this fixes:** the missing bootstrap CI, lambda sweep, proxy g, and a
grid-consistent learning curve for SST-2 (BERT).

**Runtime:** about 3 minutes (BERT re-extraction of 200 test sentences + reuse
of the saved kernel). No quantum kernel recomputation.

**Upload before running:** `kernel_matrices_week3.npz` to the Colab session, or
place it in your Drive `qml_project/` folder.

In [ ]:
!pip install datasets transformers torch scikit-learn numpy --quiet

import numpy as np
import time, os, json, pickle, shutil, warnings
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score
from sklearn.metrics.pairwise import rbf_kernel
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED = 42
TAG = 'sst2_bert'
print(f"device {DEVICE} | tag {TAG}")

---
## Step 0: Drive + locate the existing kernel matrix

In [ ]:
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/qml_project'
else:
    SAVE_DIR = 'qml_project'
os.makedirs(SAVE_DIR, exist_ok=True)

def checkpoint(filename, save_fn):
    save_fn(filename)
    dst = os.path.join(SAVE_DIR, os.path.basename(filename))
    if os.path.abspath(filename) != os.path.abspath(dst):
        shutil.copy(filename, dst)
    print(f"  saved -> {dst}")

# find the kernel file: session dir, Drive root, or qml_project
def locate(fname):
    for base in ['.', '/content', SAVE_DIR, '/content/drive/MyDrive']:
        p = os.path.join(base, fname)
        if os.path.exists(p):
            return p
    return None

KER = locate('kernel_matrices_week3.npz')
assert KER is not None, "Upload kernel_matrices_week3.npz to the session or Drive first."
print(f"found kernel file: {KER}")

---
## Step 1: Load the saved kernel and training data

The file carries the quantum kernel matrices plus the training features and
labels. The test labels were never saved, which is exactly the gap this
notebook closes.

In [ ]:
z = np.load(KER)
print("keys:", z.files)
K_train = z['K_train_quantum'].astype(np.float64)
K_test  = z['K_test_quantum'].astype(np.float64)
X_tr    = z['X_tr'].astype(np.float64)
y_tr    = z['y_tr']

assert K_train.shape == (1000, 1000)
assert K_test.shape == (200, 1000)
assert X_tr.shape == (1000, 8)
print(f"K_train {K_train.shape} diag mean {np.diag(K_train).mean():.6f}")
print(f"K_test  {K_test.shape}")
print(f"X_tr range [{X_tr.min():.4f}, {X_tr.max():.4f}] | y_tr balance {np.bincount(y_tr)}")

---
## Step 2: Reconstruct the exact test split

We rebuild the SST-2 pipeline with the original seed to recover the same 200
test sentences, extract their BERT embeddings, and apply PCA and scaling fit on
the training embeddings. The critical safeguard is in Step 4: we recompute the
quantum kernel for a handful of test points and confirm they match the saved
`K_test`. If they match, the reconstructed labels provably correspond to the
saved kernel.

In [ ]:
# --- rebuild SST-2 exactly as the pilot did ---
print("loading SST-2...")
try:
    ds = load_dataset("nyu-mll/glue", "sst2")
except Exception:
    ds = load_dataset("glue", "sst2")

N_TRAIN_EXTRACT = 1100
N_VAL_EXTRACT   = 250
train_data = ds['train'].shuffle(seed=SEED).select(range(N_TRAIN_EXTRACT))
val_data   = ds['validation'].shuffle(seed=SEED).select(range(N_VAL_EXTRACT))

train_sentences = [x['sentence'] for x in train_data]
train_labels    = np.array([x['label'] for x in train_data])
val_sentences   = [x['sentence'] for x in val_data]
val_labels      = np.array([x['label'] for x in val_data])
print(f"extracted {len(train_sentences)} train, {len(val_sentences)} val")

In [ ]:
# --- BERT embeddings (bert-base-uncased, CLS token, no normalization) ---
MODEL_NAME = "bert-base-uncased"
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
bert = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE).eval()

@torch.no_grad()
def embed(sentences, batch=32):
    out = []
    for i in range(0, len(sentences), batch):
        inp = tok(sentences[i:i+batch], padding=True, truncation=True,
                  max_length=128, return_tensors='pt').to(DEVICE)
        h = bert(**inp).last_hidden_state[:, 0, :]      # CLS token
        out.append(h.cpu().numpy())
    return np.vstack(out)

print("extracting embeddings...")
t0 = time.time()
train_emb = embed(train_sentences)
val_emb   = embed(val_sentences)
print(f"done in {time.time()-t0:.0f}s | train {train_emb.shape} val {val_emb.shape}")

In [ ]:
# --- PCA-8 + MinMax[0,pi] fit on train, build the 1000/200 subset ---
pca = PCA(n_components=8, random_state=SEED).fit(train_emb)
scaler = MinMaxScaler(feature_range=(0, np.pi)).fit(pca.transform(train_emb))

Xtr_all = np.clip(scaler.transform(pca.transform(train_emb)), 0, np.pi)
Xval_all = np.clip(scaler.transform(pca.transform(val_emb)), 0, np.pi)

idx_tr = np.random.RandomState(SEED).choice(len(Xtr_all), 1000, replace=False)
idx_te = np.random.RandomState(SEED).choice(len(Xval_all), 200, replace=False)

X_tr_rebuilt = Xtr_all[idx_tr]
y_tr_rebuilt = train_labels[idx_tr]
X_te         = Xval_all[idx_te]
y_te         = val_labels[idx_te]
print(f"rebuilt subset: X_tr {X_tr_rebuilt.shape}, X_te {X_te.shape}")
print(f"y_te balance {np.bincount(y_te)}")

---
## Step 3: Verify the rebuilt training features match the saved kernel

Before trusting the reconstructed test labels, confirm the rebuilt training
features are the same ones used to compute the saved kernel. If the max
absolute difference is tiny, the pipeline reproduced the original split
exactly.

In [ ]:
diff = np.abs(np.sort(X_tr_rebuilt, axis=0) - np.sort(X_tr, axis=0)).max()
print(f"max abs diff between rebuilt and saved X_tr (sorted): {diff:.2e}")
train_matches = diff < 1e-6
print("TRAIN FEATURES MATCH" if train_matches else "TRAIN FEATURES DIFFER -- see note below")

# direct (unsorted) check too
diff_direct = np.abs(X_tr_rebuilt - X_tr).max()
print(f"max abs diff (direct order): {diff_direct:.2e}")

---
## Step 4: The decisive check — recompute a few test kernel columns

We recompute the quantum kernel between the first few reconstructed test points
and the training set, and compare against the saved `K_test`. This is the proof
that the reconstructed `y_te` belongs to the saved kernel. We only need a small
sample: computing 5 test rows is seconds, not minutes.

In [ ]:
import pennylane as qml
DEV = qml.device('lightning.qubit', wires=8)

def iqp_feature_map(x, wires):
    n = len(wires)
    for w in wires: qml.Hadamard(wires=w)
    for i, w in enumerate(wires): qml.RZ(x[i], wires=w)
    for i in range(n - 1):
        qml.CNOT(wires=[wires[i], wires[i+1]])
        qml.RZ(x[i]*x[i+1], wires=wires[i+1])
        qml.CNOT(wires=[wires[i], wires[i+1]])

@qml.qnode(DEV)
def kcirc(a, b):
    iqp_feature_map(a, wires=range(8))
    qml.adjoint(iqp_feature_map)(b, wires=range(8))
    return qml.probs(wires=range(8))

qk = lambda a, b: float(kcirc(a, b)[0])

# recompute 5 test rows against the saved training features
CHECK_ROWS = 5
recomputed = np.array([[qk(X_te[i], X_tr[j]) for j in range(X_tr.shape[0])]
                       for i in range(CHECK_ROWS)])
saved = K_test[:CHECK_ROWS]
row_err = np.abs(recomputed - saved).max()
print(f"max abs diff on {CHECK_ROWS} recomputed test rows vs saved K_test: {row_err:.2e}")

kernel_matches = row_err < 1e-4
if kernel_matches:
    print("\nVERIFIED: reconstructed test set matches the saved kernel.")
    print("The reconstructed y_te provably corresponds to K_test.")
else:
    print("\nMISMATCH: the reconstructed test split does not match the saved kernel.")
    print("Do NOT trust the CI below. See the recovery note at the end.")

---
## Step 5: Classical baselines and the utility gap

Train the three classical SVMs on the reconstructed features and confirm the
quantum F1 from the saved kernel matches the pilot's recorded value
(0.6550).

In [ ]:
def bf1(yt, yp): return float(f1_score(yt, yp, average='binary'))

baselines, fitted = {}, {}
for name, clf in [('Linear', SVC(kernel='linear', C=1.0, random_state=SEED)),
                  ('RBF', SVC(kernel='rbf', C=1.0, gamma='scale', random_state=SEED)),
                  ('Polynomial', SVC(kernel='poly', degree=3, coef0=1, C=1.0, gamma='scale', random_state=SEED))]:
    yp = clf.fit(X_tr, y_tr).predict(X_te)
    fitted[name] = clf
    baselines[name] = {'accuracy': float(accuracy_score(y_te, yp)), 'f1': bf1(y_te, yp)}
    print(f"  {name:<12} f1={baselines[name]['f1']:.4f}")
best_nm = max(baselines, key=lambda k: baselines[k]['f1'])
best_f1 = baselines[best_nm]['f1']

qsvm = SVC(kernel='precomputed', C=1.0).fit(K_train, y_tr)
y_pred = qsvm.predict(K_test)
f1_q = bf1(y_te, y_pred); acc_q = float(accuracy_score(y_te, y_pred))
gap = f1_q - best_f1
print(f"\nquantum F1={f1_q:.4f} (pilot recorded 0.6550) | best classical {best_f1:.4f} ({best_nm})")
print(f"gap = {gap:+.4f}")

---
## Step 6: All diagnostics (identical definitions to Section 3.5)

In [ ]:
def kta(K, y):
    y = np.asarray(y); n = len(y)
    Y = np.zeros((n, n))
    for c in np.unique(y):
        m = (y == c).astype(float); Y += np.outer(m, m)
    H = np.eye(n) - np.ones((n, n))/n
    Kc, Yc = H @ K @ H, H @ Y @ H
    return float(np.sum(Kc*Yc) / np.sqrt(np.sum(Kc*Kc) * np.sum(Yc*Yc)))

def concentration(K):
    mu = np.linalg.eigvalsh(K); mu = np.maximum(mu, 0)
    lam = mu / mu.sum()
    return float(np.exp(-np.sum(lam * np.log(lam + 1e-15))) / K.shape[0])

def g_exact(K_Q, K_C, lam=1e-6):
    n = K_Q.shape[0]
    K_Q = K_Q * n / np.trace(K_Q); K_C = K_C * n / np.trace(K_C)
    w, V = np.linalg.eigh(K_Q); w = np.maximum(w, 0)
    sq = (V * np.sqrt(w)) @ V.T
    return float(np.sqrt(max(np.linalg.eigvalsh(sq @ np.linalg.inv(K_C + lam*n*np.eye(n)) @ sq).max(), 0.0)))

def g_proxy(K_Q, K_C):
    eff = lambda K: (np.trace(K)**2) / (np.sum(K**2) + 1e-10)
    return float(eff(K_Q) / (eff(K_C) + 1e-10))

Krbf = rbf_kernel(X_tr, X_tr, gamma=1.0/X_tr.shape[1])
A_Q, A_C = kta(K_train, y_tr), kta(Krbf, y_tr)
rho_ = concentration(K_train)
g_ex = g_exact(K_train, Krbf)
g_px = g_proxy(K_train, Krbf)
lam_sweep = {f"{l:.0e}": round(g_exact(K_train, Krbf, l), 2) for l in [1e-4, 1e-5, 1e-6, 1e-7]}
off_q = K_train[~np.eye(1000, dtype=bool)].mean()
off_c = Krbf[~np.eye(1000, dtype=bool)].mean()

print(f"A_Q={A_Q:.4f} (pilot 0.0212) | A_C={A_C:.4f} (pilot 0.0390)")
print(f"rho={rho_:.4f} | g={g_ex:.2f} | proxy={g_px:.2f}")
print(f"off-diag quantum {off_q:.4f} | rbf {off_c:.4f}")
print(f"lambda sweep: {lam_sweep}")

---
## Step 7: Bootstrap CI (the field that was missing)

In [ ]:
rng = np.random.RandomState(SEED)
yc_pred = fitted[best_nm].predict(X_te)
boot = []
for _ in range(1000):
    idx = rng.randint(0, len(y_te), len(y_te))
    if len(np.unique(y_te[idx])) < 2: continue
    boot.append(bf1(y_te[idx], y_pred[idx]) - bf1(y_te[idx], yc_pred[idx]))
boot = np.array(boot)
ci_lo, ci_hi = np.percentile(boot, [2.5, 97.5])
excludes_zero = bool(ci_lo > 0 or ci_hi < 0)
print(f"gap {gap:+.4f}  95% CI [{ci_lo:+.4f}, {ci_hi:+.4f}]  excludes zero: {excludes_zero}")
print(f"({len(boot)} valid resamples)")

---
## Step 8: Grid-consistent learning curve (reuses the saved kernel)

In [ ]:
sizes = [50, 100, 200, 400, 700, 1000]
lc = {'Quantum (IQP)': [], 'Linear': [], 'RBF': []}
for nn in sizes:
    yp = SVC(kernel='precomputed', C=1.0).fit(K_train[:nn,:nn], y_tr[:nn]).predict(K_test[:,:nn])
    lc['Quantum (IQP)'].append(bf1(y_te, yp))
    lc['Linear'].append(bf1(y_te, SVC(kernel='linear', C=1.0, random_state=SEED).fit(X_tr[:nn], y_tr[:nn]).predict(X_te)))
    lc['RBF'].append(bf1(y_te, SVC(kernel='rbf', C=1.0, gamma='scale', random_state=SEED).fit(X_tr[:nn], y_tr[:nn]).predict(X_te)))
print("learning curve (F1):")
for i, nn in enumerate(sizes):
    print(f"  N={nn:>4}  Q={lc['Quantum (IQP)'][i]:.4f}  Lin={lc['Linear'][i]:.4f}  RBF={lc['RBF'][i]:.4f}")

---
## Step 9: Save the complete, grid-parity record

In [ ]:
record = {
    'tag': TAG, 'domain': 'text', 'encoder': 'BERT', 'dataset': 'sst2_binary',
    'metric': 'binary', 'n_qubits': 8, 'n_train': 1000, 'n_test': 200,
    'quantum_accuracy': acc_q, 'quantum_f1': f1_q,
    'best_classical_f1': best_f1, 'best_classical_kernel': best_nm, 'gap': gap,
    'kta': A_Q, 'c_kta': A_C, 'concentration': rho_,
    'geometric_difference_exact': g_ex, 'geometric_difference_proxy': g_px,
    'exact_g_lambda_sweep': lam_sweep,
    'offdiag_quantum': float(off_q), 'offdiag_rbf': float(off_c),
    'ci_lo': float(ci_lo), 'ci_hi': float(ci_hi), 'excludes_zero': excludes_zero,
    'classical_baselines': baselines,
    'learning_curve_sizes': sizes, 'learning_curve_results': lc,
    'reconstruction_verified': bool(kernel_matches),
}
# also save the reconstructed test labels so this never has to be redone
checkpoint(f'{TAG}_test_labels.npz',
           lambda p: np.savez(p, X_te=X_te, y_te=y_te))
checkpoint(f'{TAG}_grid_record.json',
           lambda p: json.dump(record, open(p, 'w'), indent=2))

print(json.dumps({k: v for k, v in record.items() if k != 'classical_baselines'}, indent=2))
if kernel_matches:
    print("\nCOMPLETE: SST-2 (BERT) now has every field the other configs have.")
else:
    print("\nWARNING: reconstruction did not verify. CI may be unreliable; see note.")

---
## If Step 4 did not verify

If the recomputed test rows did not match the saved `K_test`, the SST-2
validation set or shuffle behavior changed between library versions, so the
reconstructed order differs from the pilot. Two options:

1. **Recompute the full test kernel** (about 15 minutes for 200x1000, far less
   than the 83-minute training kernel). Replace `K_test` with the recomputed
   matrix built from the reconstructed `X_te`; then every number below is
   internally consistent and correct, and the saved training kernel is still
   reused. To do this, set `RECOMPUTE_KTEST = True` in the cell below and re-run
   from Step 5.
2. Locate the original `embeddings_week2.npz` from the pilot, which stored
   `y_te_subset` directly, and load `y_te` from it instead of reconstructing.

In [ ]:
RECOMPUTE_KTEST = False   # set True only if Step 4 failed to verify
if RECOMPUTE_KTEST:
    print("recomputing full test kernel (~15 min)...")
    t0 = time.time()
    K_test = qml.kernels.kernel_matrix(X_te, X_tr, qk)
    print(f"done in {(time.time()-t0)/60:.1f} min; now re-run Step 5 onward")
    checkpoint('kernel_matrices_week3_verified.npz',
               lambda p: np.savez(p, K_train_quantum=K_train, K_test_quantum=K_test,
                                  X_tr=X_tr, y_tr=y_tr, X_te=X_te, y_te=y_te))